# Explore LLM Output Parquet Files

This notebook makes it easier to inspect wide QA rows and long text fields.

In [21]:
from pathlib import Path
import os

import pandas as pd
import pyarrow.dataset as ds
from IPython.display import display


In [22]:
# Update these paths for your run outputs.
SCRATCH = Path(os.environ["SCRATCH"])
QA_DIR = SCRATCH / "servicenow_incidents_processed/qa_parquet"
LLM_REJECTED_DIR = SCRATCH /  "servicenow_incidents_processed/llm_rejected_parquet/"


In [23]:
# Display settings for wide tables and long text.
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 2000)
pd.set_option("display.width", 240)
pd.set_option("display.expand_frame_repr", False)


In [24]:
def read_parquet_dir(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Path not found: {path}")
    table = ds.dataset(str(path), format="parquet").to_table()
    return table.to_pandas()


qa_df = read_parquet_dir(QA_DIR)
llm_rejected_df = read_parquet_dir(LLM_REJECTED_DIR)


In [25]:
print(f"qa rows: {len(qa_df):,}")
print(f"llm_rejected rows: {len(llm_rejected_df):,}")
print(f"total rows: {len(qa_df) + len(llm_rejected_df):,}")

qa rows: 382
llm_rejected rows: 609
total rows: 991


In [26]:
print("QA columns:")
for c in qa_df.columns:
    print("-", c)

print("\nLLM-rejected columns:")
for c in llm_rejected_df.columns:
    print("-", c)

QA columns:
- source_path
- incident_number
- sys_id
- state
- category
- subcategory
- u_resource
- short_description
- close_code
- qa_question
- qa_answer
- qa_summary
- qa_tags
- quality_reason

LLM-rejected columns:
- source_path
- quality_score
- quality_reason
- quality_raw


## Focused QA view

In [33]:
# Focused QA view (edit this list as needed).
qa_view_cols = [
    "incident_number",
    "short_description",
    "qa_question",
    "qa_answer",
    "qa_summary",
    "qa_tags",
    "quality_reason",
]

qa_view_cols = [c for c in qa_view_cols if c in qa_df.columns]
display(qa_df[qa_view_cols].head(3))

,incident_number,short_description,qa_question,qa_answer,qa_summary,qa_tags,quality_reason
0,INC0171536,Shifter memory limits,"I'm running a shifter container on an exclusive interactive node (Cori Haswell) with `salloc -N 1 --qos interactive --image=registry.services.nersc.gov/imagename -t 01:01:00 --exclusive --mem=100G -C haswell shifter /bin/bash`. When I allocate a large NumPy array (e.g., `t = np.zeros(int(1e9))`) and then do operations like `t+t` or `.copy()`, the job is killed with a `Killed` OOM message, even though the node has ~130 GB RAM. I tried `--mem=0`, different images (docker:slothai/numpy), and checked `ulimit` (unlimited). Why am I getting OOM so early and how can I fix it?","The OOM kills were caused by running shifter directly in the salloc command line, which started the container on the MOM node where cgroup memory limits (~2 GB) applied. The fix is to allocate the interactive node first with salloc and then launch shifter inside that allocation, e.g. `$ salloc -N 1 --qos interactive -t 01:01:00 --exclusive --mem=0 -C haswell` followed by `shifter --image=<image> /path/to/python -c 'import numpy as np; t=np.zeros(int(1e9)); t+t;'`. Using `--mem=100G` or `--mem=0` in the original salloc command, or trying different containers, did not prevent the OOM kills.","Running shifter directly in the salloc command placed the container on the MOM node with cgroup memory limits, causing early OOM; allocating the node first with salloc and then launching shifter inside the allocation resolves the issue.","[shifter, OOM, salloc, memory limits, cgroup, numpy, interactive node]","Clear OOM issue, reproducible steps, detailed fix with two‑step salloc/shifter, good technical depth."
1,INC0171565,Problems during using jupyterlab,"I'm using JupyterLab on NERSC and can't access many tools. In the launcher I only see Python 3, no conda, vim, Julia, PyROOT, etc. Also, when I browse /opt inside JupyterLab I only see 'cray', 'mods', and 'udiImage', but via SSH I see many more directories like IBM, R, gcc, intel, python, etc. What could be causing this limited view and missing applications?","The issue was caused by a custom profile set in the JupyterHub profile service, which runs JupyterLab inside a Shifter container and therefore shows a limited /opt directory and only a single default kernel. The fix was to deselect this profile in the JupyterHub profile service, after which the default /opt layout and the full set of kernels (conda, vim, Julia, PyROOT, etc.) reappeared in the launcher.",The limited /opt view and missing kernels in JupyterLab were due to a custom profile running a Shifter container; deselecting the profile in JupyterHub restored the default environment.,"[JupyterLab, JupyterHub, Shifter, profile service, kernel, /opt, launcher]","Clear problem, moderate solution detail, good technical context, but steps lack specifics."
2,INC0171681,Need help with VISIT for graphics,"I'm trying to make a surface plot on the x‑z plane of my scalar real4 data (Eyf(lx,lz) with lx=lz=512) stored as .bov and .dat files on Perlmutter. I can make pseudocolor plots in VisIt, but I can't get a surface plot. I tried rewriting the BOV header to X,Y,Z order and even writing a 3‑D cube of data, but still no surface plot. Is there a way in VisIt to produce this surface plot, and can I change the color map?","The surface plot was achieved by applying VisIt’s Elevation Operator to the 2‑D data, which creates the desired x‑z plane surface without modifying the .bov/.dat files. Color maps can be changed via the Controls → Color Tables menu (or by creating custom tables). The earlier suggestions to rewrite the BOV header to X,Y,Z order and to write a full 3‑D cube were tried but were not needed for the surface plot.",The user needed a surface plot of 2‑D data in VisIt; the issue was resolved by using the Elevation Operator (no BOV changes required) and color maps can be adjusted via Controls → Color Tables.,"[VisIt, BOV, surface plot, Elevatio

## LLM-rejected focused view.

In [32]:
# LLM-rejected focused view.
rej_view_cols = [
    "source_path",
    "quality_score",
    "quality_reason",
    "quality_raw",
]

rej_view_cols = [c for c in rej_view_cols if c in llm_rejected_df.columns]
display(llm_rejected_df[rej_view_cols].head(3))

,source_path,quality_score,quality_reason,quality_raw
0,INC016/83/01.json,55,Specific issue but solution lacks detail and technical depth.,"analysisWe need to score the ticket. The ticket is about fixing a broken URL on NERSC website. It's not about decommissioned system. Problem clarity: specific broken link, gives old URL and correct URL. That's clear. Score maybe 25/30. Solution completeness: resolution says ""fixed broken link on nersc webpage"". Not detailed commands, but simple fix. For QA training, maybe not very deep. Score maybe 20/40. Technical depth: low, just a link change. Score maybe 5/20. Reproducibility: others could follow steps to edit webpage, but not detailed. Score maybe 5/10. Total = 25+20+5+5 = 55. That's below 70, so keep false. Provide JSON with score 55, keep false, reason brief. Under 20 words. Reason: ""Specific issue but solution lacks detail and technical depth."" That's 9 words. Return JSON.assistantfinal{""keep"": false, ""reason"": ""Specific issue but solution lacks detail and technical depth."", ""score"": 55}"
1,INC016/83/02.json,58,Clear issue but solution only provides contact info; lacks technical depth and detailed steps.,"analysisWe need to score the ticket. Evaluate criteria:\n\nProblem clarity (30): The issue is ""help@nersc.gov is not valid?"" The user reports email bounce. That's fairly clear. Specific error: unknown user ""help"". Good.\n\nSolution completeness (40): Resolution: ""provided staff user with nersc contact email info"". The solution is just giving correct email addresses. No detailed steps, but it's a simple informational answer. It's complete enough? The solution is minimal but resolves the issue. However not many commands. Might be considered low completeness.\n\nTechnical depth (20): Very low depth, just email address info. No configs.\n\nReproducibility (10): Others can reproduce by sending email to that address. The ticket includes bounce message. So reproducible.\n\nScore: Problem clarity 25/30. Solution completeness maybe 20/40. Technical depth 5/20. Reproducibility 8/10. Total = 58. That's <70, so not keep.\n\nAlso not about decommissioned system. So keep false.\n\nReturn JSON with score 58, keep false, reason brief (<20 words). Reason: ""Clear issue but solution only provides contact info; lacks technical depth and detailed steps."" That's 14 words.\n\nReturn JSON.assistantfinal{""keep"": false, ""reason"": ""Clear issue but solution only provides contact info; lacks technical depth and detailed steps."", ""score"": 58}"
2,INC016/83/66.json,20,"Ticket focuses on Cori, a decommissioned system","analysisWe need to score the ticket for QA training usefulness. The ticket is about asking which OpenSHMEM version to use on Cori, a decommissioned system (Cori). The instruction says automatic reject (score <=30) if ticket is primarily about a decommissioned NERSC system: Cori, KNL, Haswell (Cori partition), or Edison. Here the subject is about using OpenSHMEM on Cori, asking which version to use. That is the main subject, so it's about a decommissioned system. Therefore we must reject automatically, score <=30. Should we give a score? Probably low, maybe 20. Keep false. Reason: decommissioned system focus. Must output JSON only.assistantfinal{""keep"": false, ""reason"": ""Ticket focuses on Cori, a decommissioned system"", ""score"": 20}"


## Single-row detail view

In [29]:
# Single-row detail view (transposed) for full readability.
row_idx = 0  # change this index
display(qa_df.iloc[[row_idx]].T)

,0
source_path,INC017/15/36.json
incident_number,INC0171536
sys_id,c148a2af1b2db01059b0dc6ce54bcbde
state,Closed
category,Running Jobs
subcategory,
u_resource,
short_description,Shifter memory limits
close_code,Resolved by Requester
qa_question,"I'm running a shifter container on an exclusive interactive node (Cori Haswell) with `salloc -N 1 --qos interactive --image=registry.services.nersc.gov/imagename -t 01:01:00 --exclusive --mem=100G -C haswell shifter /bin/bash`. When I allocate a large NumPy array (e.g., `t = np.zeros(int(1e9))`) and then do operations like `t+t` or `.copy()`, the job is killed with a `Killed` OOM message, even though the node has ~130 GB RAM. I tried `--mem=0`, different images (docker:slothai/numpy), and checked `ulimit` (unlimited). Why am I getting OOM so early and how can I fix it?"


In [19]:
# Human-readable QA ticket inspector (non-table output).
from html import escape
from IPython.display import display, HTML
import json

ticket_idx = 0
ticket = qa_df.iloc[ticket_idx]

def normalize(v):
    if v is None:
        return None
    if isinstance(v, (list, tuple, set)):
        return list(v)
    if hasattr(v, "tolist") and not isinstance(v, (str, bytes)):
        try:
            return v.tolist()
        except Exception:
            pass
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return v

def parse_original_ticket_json(v):
    v = normalize(v)
    if v is None:
        return None
    if isinstance(v, dict):
        return v
    if isinstance(v, str):
        s = v.strip()
        if not s:
            return None
        try:
            parsed = json.loads(s)
            return parsed if isinstance(parsed, dict) else {"_value": parsed}
        except Exception:
            return {"_raw": s}
    return {"_value": v}

def block_text(v):
    v = normalize(v)
    if v is None:
        return "<span style='color:#888'>[missing]</span>"
    if isinstance(v, list):
        if not v:
            return "<span style='color:#888'>[missing]</span>"
        items = "".join(f"<li>{escape(str(x))}</li>" for x in v if x is not None)
        return f"<ul>{items}</ul>" if items else "<span style='color:#888'>[missing]</span>"
    return f"<pre style='white-space:pre-wrap; margin:0'>{escape(str(v).strip())}</pre>"

fields = {
    "Incident Number": ticket.get("incident_number"),
    "Short Description": ticket.get("short_description"),
    "Question": ticket.get("qa_question"),
    "Answer": ticket.get("qa_answer"),
    "Summary": ticket.get("qa_summary"),
    "Tags": ticket.get("qa_tags"),
    "Quality Reason": ticket.get("quality_reason"),
    "Source Path": ticket.get("source_path"),
}

cards = []
for k, v in fields.items():
    cards.append(f"""
<div style="margin:10px 0; padding:10px 12px; border:1px solid #ddd; border-radius:8px;">
  <div style="font-weight:700; margin-bottom:6px;">{escape(k)}</div>
  {block_text(v)}
</div>
""")

original_ticket = parse_original_ticket_json(ticket.get("original_ticket_json"))
if original_ticket is None:
    original_ticket_block = "<span style='color:#888'>[missing]</span>"
else:
    original_ticket_block = escape(json.dumps(original_ticket, indent=2, default=str))

display(HTML(f"""
<div style="font-family: ui-sans-serif, system-ui; max-width: 1100px;">
  <h3 style="margin-bottom:4px;">QA Ticket Inspection</h3>
  <div style="color:#555; margin-bottom:12px;">row index: {ticket_idx}</div>
  {''.join(cards)}
  <details style="margin-top:12px;">
    <summary style="cursor:pointer;">Show original ticket JSON</summary>
    <pre style="white-space:pre-wrap;">{original_ticket_block}</pre>
  </details>
  <details style="margin-top:12px;">
    <summary style="cursor:pointer;">Show full QA row JSON</summary>
    <pre style="white-space:pre-wrap;">{escape(json.dumps({k: normalize(v) for k, v in ticket.to_dict().items()}, indent=2, default=str))}</pre>
  </details>
</div>
"""))
